# Single-Element DG Advection: Dynamic Wave Propagation Solver

**Status:** ✅ **TIME-DEPENDENT TRANSCENDENTAL WAVE TEST** (2026-04-09)

**Reference Framework:** Hesthaven-Warburton (HW) standard **[-1,1]×[-1,1]** simplex

**Test Case:** Time-dependent non-polynomial wave propagation to validate spatial truncation error convergence

## 🎯 Key Differences from Stationary Test

### ✅ Physics Updates

| Feature | Stationary | Dynamic | Status |
|---------|-----------|---------|--------|
| **Exact Solution** | $q_\text{exact} = x - y$ (polynomial) | $q_\text{exact} = \sin(x - t)$ (transcendental) | ✅ Non-polynomial |
| **Velocity Field** | $\boldsymbol{V} = (1, 1)$ | $\boldsymbol{V} = (1, 0)$ | ✅ Pure x-advection |
| **Initial Condition** | Hardcoded: $q(x,y,0) = x-y$ | Dynamic: $q(x,y,0) = \sin(x)$ | ✅ From exact_solution() |
| **BC Behavior** | Perfect (zero error) | Transcendental approximation | ✅ Truncation error |
| **Time Loop Exit** | Early convergence @ t=0.011 | Always reaches t_final | ✅ Full integration |

## 🔬 Problem Formulation

**Scalar Transport Equation (Strong Form):**
$$\frac{\partial q}{\partial t} + \boldsymbol{V} \cdot \nabla q = 0$$

where $\boldsymbol{V} = (u, v) = (1, 0)$ (pure x-direction advection).

**Analytical Solution (Dynamic Wave Test):**
$$q_{\text{exact}}(x, y, t) = \sin(x - t) \quad \text{(non-polynomial)}$$

**Mathematical Verification:**
$$\frac{\partial q}{\partial t} = -\cos(x-t), \quad \frac{\partial q}{\partial x} = \cos(x-t)$$
$$\frac{\partial q}{\partial t} + \frac{\partial q}{\partial x} = -\cos(x-t) + \cos(x-t) = 0 \,\checkmark$$

## 📐 DG Split-Form Formulation

**Volume Integral (Skew-Symmetric):**
$$\text{volume} = -\frac{1}{2}\left[D_x(u \odot q) + D_y(v \odot q) + u \odot D_x q + v \odot D_y q + (\nabla \cdot \boldsymbol{V}) \odot q\right]$$

**Surface Integral (Branchless Upwind Flux):**
$$\text{surface} = E^T W_e \left[\frac{1}{2}(v_n - |v_n|)(q_{\text{bound}} - q_{\text{exact}})\right]$$

**Time Integration (LSRK54 - Carpenter & Kennedy 1994):**
$$du = A_{\text{RK}}[k] \cdot du + \Delta t \cdot R(q)$$
$$q_{n+1} = q_n + B_{\text{RK}}[k] \cdot du$$

## 🎓 Expected Error Behavior

### Spatial Truncation Error: $O(h^{k+1})$

Since $\sin(x-t)$ is **transcendental** (not a polynomial), DG methods cannot represent it exactly, even with high polynomial degree $k$. Instead, the solution is approximated via nodal interpolation, leading to:

$$\|q - q_{\text{computed}}\|_2 \approx C \cdot h^{k+1}$$

where:
- $h$ = characteristic mesh size (here: element diameter)
- $k$ = polynomial degree ($k=3$ in this test)
- $C$ = constant depending on solution smoothness and geometry

**For this test:**
- Polynomial degree: $k = 3$, so expect $O(h^4)$ convergence
- Nodes per element: $(k+1)^2 = 16$
- Characteristic h: diameter of inscribed circle
- **Expected L2 error: ~$10^{-3}$ to $10^{-4}$ (interpolation error, not machine precision)**

### Why NOT Machine Precision?

1. **Interpolation Error:** The sine wave must be approximated by piecewise polynomials → finite error
2. **No Exact Solution:** Unlike $q=x-y$ (exactly representable), $\sin(x)$ requires infinite series → truncation
3. **Time Integration Error:** LSRK54 is 5th order, but spatial error dominates for this coarse mesh

This test **validates that the solver correctly implements the DG method**, because the L2 error will match theoretical predictions for transcendental functions.

## 🏗️ Implementation Architecture

| Cell | Component | Implementation | API Integration |
|------|-----------|-----------------|------------------|
| **0** | Physical Setup | Edge normals, vertices, CCW validation | Native |
| **1** | Reference Data | HW coordinates (r,s), weights extraction | `get_reference_data('table1', k)` ✅ |
| **2** | Operators & Boundaries | Vandermonde, D matrices, fmask | `build_fmask_table1()`, `build_differentiation_matrices()` ✅ |
| **3** | RHS (Split Form) | Physics functions + generalized flux | **NEW:** `sin(x-t)` wave & `u=1,v=0` |
| **4** | Time Stepping | LSRK54 (5 stages, residual storage) | **MODIFIED:** Dynamic IC, no early convergence |
| **5** | Validation | L2 error measurement | **UPDATED:** Spatial truncation analysis |

## 📊 Key Physics Functions (Updated)

```python
def exact_solution(x, y, t):
    """Exact solution: transcendental sine wave"""
    return np.sin(x - t)

def velocity_field(x, y, t):
    """Velocity field: pure x-direction advection"""
    u = np.ones_like(x)
    v = np.zeros_like(x)   # v=0 ensures ∂q/∂t + ∂q/∂x = 0
    return u, v
```

## 📚 References

1. Cockburn, B., Karniadakis, G. E., & Shu, C. W. (2000). *Runge-Kutta Discontinuous Galerkin Methods*. Lect. Notes Comput. Sci. Eng. 11.
2. Carpenter, M. H., & Kennedy, C. A. (1994). Fourth-order 2N-storage Runge-Kutta schemes. NASA TM 109111.
3. Dubiner, M. (1991). Spectral methods on triangles and other domains. J. Sci. Comput., 6(4), 345-390.
4. Hesthaven, J. S., & Warburton, T. (2008). *Nodal DG Methods*. Springer.

In [10]:
# ============================================================================
# CELL 0: Initialization & Imports
# ============================================================================

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Tuple
import sys

# Add src to path (handle both interactive and script execution)
notebook_dir = Path("/Users/user/code/Simplex-DG-solver/notebooks/experimental")
project_root = notebook_dir.parent.parent
sys.path.insert(0, str(project_root))

# Import core utilities
from src.core.generators import get_reference_data
from src.geometry.metrics import compute_geometric_factors

# ============================================================================
# REFERENCE ELEMENT STANDARD
# ============================================================================

# This notebook uses the HESTHAVEN-WARBURTON STANDARD reference element:
# Reference domain: [-1, 1] × [-1, 1] simplex with vertices:
#   v_ref_1 = (-1, -1)  [bottom-left]
#   v_ref_2 = ( 1, -1)  [bottom-right]
#   v_ref_3 = (-1,  1)  [top-left]
#
# Physical mapped element defined in Cell 0 (see v1, v2, v3 below)

# ============================================================================
# STEP 1: Define Physical Element (Single Triangle)
# ============================================================================

# Physical element vertices: triangle (0,0), (2,0), (0,2)
# This forms a right triangle with area = 0.5 * 2 * 2 = 2.0
v1 = np.array([0.0, 0.0])  # Vertex 1 (maps to reference (-1,-1))
v2 = np.array([2.0, 0.0])  # Vertex 2 (maps to reference (1,-1))
v3 = np.array([0.0, 2.0])  # Vertex 3 (maps to reference (-1,1))

vertices = np.array([v1, v2, v3])  # Array of all vertices

# Compute outward-pointing normals for each edge (from CCW orientation)
# For a triangle with CCW vertices, outward normal = perpendicular to edge, pointing outward
# Edge i: vertices[i] → vertices[(i+1)%3]
edge_0 = v2 - v1  # (2, 0)
edge_1 = v3 - v2  # (-2, 2)
edge_2 = v1 - v3  # (0, -2)

# Rotate 90° clockwise for outward normal: (x, y) → (y, -x)
n0 = np.array([edge_0[1], -edge_0[0]])  # (0, -2) → normalized: (0, -1)
n1 = np.array([edge_1[1], -edge_1[0]])  # (2, 2) → normalized: (1/√2, 1/√2)
n2 = np.array([edge_2[1], -edge_2[0]])  # (-2, 0) → normalized: (-1, 0)

# Normalize to unit vectors
normals = np.array([n0 / np.linalg.norm(n0),
                    n1 / np.linalg.norm(n1),
                    n2 / np.linalg.norm(n2)])

print(f"Physical element setup:")
print(f"  Vertices: v1={v1}, v2={v2}, v3={v3}")
print(f"  Reference element: Hesthaven-Warburton [-1,1]×[-1,1] simplex")
print(f"  Edge normals (outward, unit):")
print(f"    Edge 0: {normals[0]}")
print(f"    Edge 1: {normals[1]}")
print(f"    Edge 2: {normals[2]}")

Physical element setup:
  Vertices: v1=[0. 0.], v2=[2. 0.], v3=[0. 2.]
  Reference element: Hesthaven-Warburton [-1,1]×[-1,1] simplex
  Edge normals (outward, unit):
    Edge 0: [ 0. -1.]
    Edge 1: [0.70710678 0.70710678]
    Edge 2: [-1. -0.]


In [11]:
# ============================================================================
# CELL 1: Reference Element ([-1,1]×[-1,1] Hesthaven-Warburton Standard)
# ============================================================================

# Polynomial degree (k = 3 corresponds to quadratic basis)
k = 4
ref_data = get_reference_data('table1', k)

# Extract reference element data in NATIVE xi, eta coordinates
# These are already in [-1,1]×[-1,1] the HW standard simplex
xi = ref_data['xi']
eta = ref_data['eta']
weights = ref_data['weights']
weights_1d = ref_data['weights_1d']  # 1D boundary weights (extracted from API)
n_nodes = len(xi)

print(f"\nReference element data (Hesthaven-Warburton [-1,1]×[-1,1]):")
print(f"  Polynomial degree: k={k}")
print(f"  Number of nodes: {n_nodes}")
print(f"  Nodes (xi, eta) range: xi in [{xi.min():.4f}, {xi.max():.4f}], eta in [{eta.min():.4f}, {eta.max():.4f}]")
print(f"  Reference vertices: (-1,-1), (1,-1), (-1,1)")
print(f"  1D boundary quadrature weights (nfp={len(weights_1d)}): {weights_1d}")

# ============================================================================
# STEP 2: Direct Assignment to Reference Coordinates (NO Barycentric Conversion)
# ============================================================================

# In the HW standard, xi and eta ARE the reference coordinates r and s
# No transformation needed—we use them directly
r = xi
s = eta

print(f"\nReference coordinates assignment:")
print(f"  r ≡ ξ (no conversion)")
print(f"  s ≡ η (no conversion)")

# ============================================================================
# STEP 3: Compute Geometric Factors using API
# ============================================================================
# Use the external compute_geometric_factors API (required for modularity)

metrics = compute_geometric_factors(v1, v2, v3)
jacobian = metrics['J']
rx, ry = metrics['rx'], metrics['ry']
sx, sy = metrics['sx'], metrics['sy']

print(f"\nGeometric factors (via compute_geometric_factors API):")
print(f"  Jacobian determinant |J| = {jacobian:.6f}")
print(f"  rx = {rx:.6f}, ry = {ry:.6f}")
print(f"  sx = {sx:.6f}, sy = {sy:.6f}")

# ============================================================================
# STEP 4: Compute Physical Coordinates using HW Affine Mapping
# ============================================================================

# Affine mapping from reference [-1,1]×[-1,1] to physical (x,y):
# x = -0.5*(r+s)*x1 + 0.5*(r+1)*x2 + 0.5*(s+1)*x3
# y = -0.5*(r+s)*y1 + 0.5*(r+1)*y2 + 0.5*(s+1)*y3

x = -0.5 * (r + s) * v1[0] + 0.5 * (r + 1) * v2[0] + 0.5 * (s + 1) * v3[0]
y = -0.5 * (r + s) * v1[1] + 0.5 * (r + 1) * v2[1] + 0.5 * (s + 1) * v3[1]
nodes_xy = np.column_stack([x, y])

print(f"\nPhysical node coordinates (first 5):")
for i in range(min(5, n_nodes)):
    print(f"  node {i}: (x, y) = ({x[i]:.4f}, {y[i]:.4f})")

print(f"\nPhysical domain: triangle with vertices v1={v1}, v2={v2}, v3={v3}")
print(f"  Element area = |J|/2 = {jacobian/2:.6f}")


Reference element data (Hesthaven-Warburton [-1,1]×[-1,1]):
  Polynomial degree: k=4
  Number of nodes: 22
  Nodes (xi, eta) range: xi in [-1.0000, 0.9062], eta in [-1.0000, 0.9062]
  Reference vertices: (-1,-1), (1,-1), (-1,1)
  1D boundary quadrature weights (nfp=5): [0.23692689 0.47862867 0.56888889 0.47862867 0.23692689]

Reference coordinates assignment:
  r ≡ ξ (no conversion)
  s ≡ η (no conversion)

Geometric factors (via compute_geometric_factors API):
  Jacobian determinant |J| = 1.000000
  rx = 1.000000, ry = -0.000000
  sx = -0.000000, sy = 1.000000

Physical node coordinates (first 5):
  node 0: (x, y) = (1.9062, 0.0000)
  node 1: (x, y) = (0.0000, 1.9062)
  node 2: (x, y) = (1.9062, 0.0938)
  node 3: (x, y) = (0.0938, 1.9062)
  node 4: (x, y) = (0.0000, 0.0938)

Physical domain: triangle with vertices v1=[0. 0.], v2=[2. 0.], v3=[0. 2.]
  Element area = |J|/2 = 0.500000


## Differentiation Operators & Boundary Extraction

In [12]:
# Cell 2: Operators (D_r, D_s, E, W) - Using build_fmask_table1 API

# ============================================================================
# STEP 0: Import boundary extraction API
# ============================================================================

from src.reconstruction.boundary import build_fmask_table1

# ============================================================================
# STEP 1: Build Differentiation Matrices using weighted mass matrix method
# ============================================================================

# Import basis functions and builder
from src.bases.vandermonde import vandermonde_2d_dubiner, grad_vandermonde_2d_dubiner
from src.reconstruction import build_differentiation_matrices

# Build Vandermonde matrix and its gradient (derivatives in reference space)
V_nodal = vandermonde_2d_dubiner(r, s, k)
Vr, Vs = grad_vandermonde_2d_dubiner(r, s, k)

print(f"Vandermonde matrix shape: {V_nodal.shape}")
print(f"  n_nodes: {V_nodal.shape[0]}")
print(f"  num_basis: {V_nodal.shape[1]}")
print(f"  Condition number of V: {np.linalg.cond(V_nodal):.2e}")

# Build differentiation matrices using weighted mass matrix method
D_r_matrix, D_s_matrix = build_differentiation_matrices(V_nodal, Vr, Vs, w=weights)

print(f"\nDifferentiation matrices via weighted mass matrix method:")
print(f"  D_r shape: {D_r_matrix.shape}, D_s shape: {D_s_matrix.shape}")
print(f"  ||D_r||_F = {np.linalg.norm(D_r_matrix):.6f}, ||D_s||_F = {np.linalg.norm(D_s_matrix):.6f}")

# ============================================================================
# STEP 2: Boundary Extraction Matrix (E) - Using build_fmask_table1 API
# ============================================================================

# Compute barycentric coordinates from HW reference frame [-1,1]×[-1,1]
# For HW simplex with vertices v1=(-1,-1), v2=(1,-1), v3=(-1,1):
#   λ1 = (-r - s) / 2
#   λ2 = (r + 1) / 2
#   λ3 = (s + 1) / 2
# These satisfy λ1 + λ2 + λ3 = 1 and correspond to:
#   Face 0 (s=-1): λ3 = 0
#   Face 1 (r=-1): λ2 = 0
#   Face 2 (r+s=0): λ1 = 0

bary_coords = np.column_stack([
    (-r - s) / 2,    # λ1
    (r + 1) / 2,     # λ2
    (s + 1) / 2      # λ3
])

# Use the modular API to extract boundary node indices for each face
fmask = build_fmask_table1(bary_coords)

print(f"\nBoundary face node counts (via build_fmask_table1 API):")
for face_id in range(3):
    face_nodes = fmask[:, face_id]
    print(f"  Face {face_id}: {len(face_nodes)} nodes")
print(f"✓ Boundary extraction completed via modular API")

# Create extraction matrix E (n_boundary × n_nodes) from fmask
nfp = fmask.shape[0]  # nodes per face
n_boundary_nodes = 3 * nfp
E = np.zeros((n_boundary_nodes, n_nodes))
boundary_idx = 0
boundary_face_map = []

for face_id in range(3):
    for local_node_idx in fmask[:, face_id]:
        E[boundary_idx, local_node_idx] = 1.0
        boundary_face_map.append(face_id)
        boundary_idx += 1

print(f"\nBoundary extraction matrix E (shape {E.shape}):")
print(f"  Total boundary nodes: {boundary_idx}")
print(f"  Matrix rank: {np.linalg.matrix_rank(E)}")

# ============================================================================
# STEP 3: Mass Matrix & Weight Diagonal
# ============================================================================

W_diag = weights  # Quadrature weights
M_diag = jacobian * weights  # Scaled by element area |J|
M_inv_diag = 1.0 / M_diag  # Inverse for time stepping

print(f"\nMass matrix:")
print(f"  Sum of |J|*weights: {np.sum(M_diag):.6f} (element area)")
print(f"  Element area = |J|/2 = {jacobian/2:.6f}")

# ============================================================================
# STEP 4: Physical Differentiation Matrices
# ============================================================================

# Chain rule: ∂q/∂x = rx * ∂q/∂r + sx * ∂q/∂s
# Matrix form: D_x = rx * D_r + sx * D_s
D_x = rx * D_r_matrix + sx * D_s_matrix
D_y = ry * D_r_matrix + sy * D_s_matrix

print(f"\nPhysical differentiation matrices:")
print(f"  D_x = {rx:.6f}*D_r + {sx:.6f}*D_s")
print(f"  D_y = {ry:.6f}*D_r + {sy:.6f}*D_s")
print(f"  ||D_x||_F = {np.linalg.norm(D_x):.6f}, ||D_y||_F = {np.linalg.norm(D_y):.6f}")

Vandermonde matrix shape: (22, 15)
  n_nodes: 22
  num_basis: 15
  Condition number of V: 4.66e+00

Differentiation matrices via weighted mass matrix method:
  D_r shape: (22, 22), D_s shape: (22, 22)
  ||D_r||_F = 29.543114, ||D_s||_F = 29.543114

Boundary face node counts (via build_fmask_table1 API):
  Face 0: 5 nodes
  Face 1: 5 nodes
  Face 2: 5 nodes
✓ Boundary extraction completed via modular API

Boundary extraction matrix E (shape (15, 22)):
  Total boundary nodes: 15
  Matrix rank: 15

Mass matrix:
  Sum of |J|*weights: 1.000000 (element area)
  Element area = |J|/2 = 0.500000

Physical differentiation matrices:
  D_x = 1.000000*D_r + -0.000000*D_s
  D_y = -0.000000*D_r + 1.000000*D_s
  ||D_x||_F = 29.543114, ||D_y||_F = 29.543114


## RHS Computation with Branchless Upwind Flux

In [13]:
# Cell 3: RHS Function - Split Form Nodal DG (Skew-Symmetric Formulation)
# ============================================================================
# UPDATED FOR DYNAMIC WAVE TEST
# ============================================================================

# ============================================================================
# PHYSICS DEFINITIONS (Updated for transcendental wave)
# ============================================================================

def exact_solution(x: np.ndarray, y: np.ndarray, t: float) -> np.ndarray:
    """Exact solution to the dynamic wave advection problem.

    For the dynamic test case: q_exact(x,y,t) = sin(x - t) (non-polynomial).
    This is a sine wave traveling in the +x direction with speed 1.0.

    Mathematical validation:
        ∂q/∂t = -cos(x-t)
        ∂q/∂x = cos(x-t)
        ∂q/∂t + ∂q/∂x = 0 ✓ (satisfies advection equation with u=1, v=0)
    """
    return np.sin(x - t)


def velocity_field(x: np.ndarray, y: np.ndarray, t: float) -> tuple[np.ndarray, np.ndarray]:
    """Velocity field components u and v (updated for pure x-advection).

    Returns (u, v) as arrays matching the shape of x, y.
    For this test: u=1 (x-advection), v=0 (no y-component).
    This ensures: ∂q/∂t + u*∂q/∂x + v*∂q/∂y = 0
    """
    u = np.ones_like(x)
    v = np.zeros_like(x)
    return u, v


def compute_rhs(q: np.ndarray, t: float, diagnostic: bool = False) -> np.ndarray:
    """Compute RHS for Split Form Nodal DG advection: dq/dt = RHS(q, t)

    Uses skew-symmetric formulation for enhanced discrete stability:
    volume_term = -0.5 * [(D_x(u*q) + D_y(v*q)) + (u*D_x(q) + v*D_y(q)) + (∇·V)*q]

    Args:
        q: Current solution vector
        t: Current time (for time-dependent problems)
        diagnostic: Print detailed term diagnostics if True
    """

    # STEP 1: Volume term (Split Form / Skew-Symmetric Formulation)
    # Dynamically evaluate velocity field at current time
    u_arr, v_arr = velocity_field(x, y, t)

    # Term 1: D_x(u*q) + D_y(v*q)
    term1 = D_x @ (u_arr * q) + D_y @ (v_arr * q)

    # Term 2: u*(D_x q) + v*(D_y q)
    term2 = u_arr * (D_x @ q) + v_arr * (D_y @ q)

    # Term 3: (D_x u + D_y v)*q
    term3 = (D_x @ u_arr + D_y @ v_arr) * q

    # Combine and scale
    volume_term = -0.5 * (term1 + term2 + term3)

    # STEP 2: Boundary extraction
    q_boundary = E @ q
    q_exact_boundary = E @ exact_solution(x, y, t)

    # STEP 3: Compute normal velocity (v_n = V · n)
    normals_expanded = np.repeat(normals, nfp, axis=0)
    v_normal = normals_expanded[:, 0] * u_arr[0] + normals_expanded[:, 1] * v_arr[0]

    # STEP 4: Branchless upwind flux
    upwind_factor = 0.5 * (v_normal - np.abs(v_normal))
    flux_penalty = upwind_factor * (q_boundary - q_exact_boundary)

    # STEP 5: Surface integral with proper 1D quadrature and face Jacobian
    # Use 1D boundary weights from the reference data API (already hardcoded in get_reference_data)
    # These are standard Gauss-Legendre weights for [-1, 1] interval

    # Replicate weights for all 3 faces
    face_w = np.tile(weights_1d, 3)

    # Compute edge lengths and face Jacobian J_face = edge_length / 2.0
    face_i_indices = np.array([0, 1, 2])
    face_j_indices = np.array([1, 2, 0])
    edge_lengths = np.linalg.norm(vertices[face_j_indices] - vertices[face_i_indices], axis=1)
    J_face = edge_lengths / 2.0
    J_face_expanded = np.repeat(J_face, nfp)

    # Scale penalty by 1D weights and face Jacobian
    scaled_penalty = flux_penalty * face_w * J_face_expanded

    # STEP 6: Surface term assembly (project boundary to volume)
    surface_term = E.T @ scaled_penalty

    # STEP 7: Combine and apply mass matrix inverse
    rhs = volume_term + (M_inv_diag * surface_term)

    # Optional diagnostic
    if diagnostic:
        print(f"  term1: [{term1.min():.6e}, {term1.max():.6e}]")
        print(f"  term2: [{term2.min():.6e}, {term2.max():.6e}]")
        print(f"  term3: [{term3.min():.6e}, {term3.max():.6e}]")
        print(f"  volume_term: [{volume_term.min():.6e}, {volume_term.max():.6e}]")
        print(f"  surface_term: [{surface_term.min():.6e}, {surface_term.max():.6e}]")
        print(f"  rhs: [{rhs.min():.6e}, {rhs.max():.6e}]")

    return rhs

In [14]:
# Quick diagnostic test: evaluate RHS at initial condition q = sin(x)
print("\n" + "="*70)
print("QUICK RHS DIAGNOSTIC TEST (Dynamic Wave)")
print("="*70)
print(f"Initial condition: q(x,y,0) = sin(x)")
print(f"Current Jacobian value: jacobian = {jacobian:.6f}")
print(f"Expected: |det([v2-v1, v3-v1])| = {np.linalg.det(np.column_stack([vertices[1]-vertices[0], vertices[2]-vertices[0]])):.6f}")
print("="*70)

q_test_initial = np.sin(x)
rhs_test = compute_rhs(q_test_initial, t=0.0, diagnostic=True)


QUICK RHS DIAGNOSTIC TEST (Dynamic Wave)
Initial condition: q(x,y,0) = sin(x)
Current Jacobian value: jacobian = 1.000000
Expected: |det([v2-v1, v3-v1])| = 4.000000
  term1: [-3.354094e-01, 9.956577e-01]
  term2: [-3.354094e-01, 9.956577e-01]
  term3: [-8.877144e-16, 4.694019e-16]
  volume_term: [-9.956577e-01, 3.354094e-01]
  surface_term: [0.000000e+00, 0.000000e+00]
  rhs: [-9.956577e-01, 3.354094e-01]


## LSRK54 Time Stepping Loop

In [15]:
# Cell 4: Initialize Time Integration Setup & Run LSRK54
# ============================================================================
# UPDATED FOR DYNAMIC WAVE TEST
# ============================================================================

# ============================================================================
# STEP 1: Setup Initial Condition & Time Parameters (DYNAMIC IC)
# ============================================================================

# CRITICAL UPDATE: Use dynamic initial condition from exact_solution at t=0
q_initial = exact_solution(x, y, t=0.0)  # q(x,y,0) = sin(x)
q = q_initial.copy()
t = 0.0
t_final = 1.0
step_count = 0

# CFL parameter for DG stability
CFL = 0.1

# ============================================================================
# DYNAMIC TIME STEP CALCULATION (CFL STABILITY CONDITION)
# ============================================================================
# dt = CFL * h_min / (V_max * N^2)
# where:
#   h_min = characteristic length (minimum altitude or inscribed circle diameter)
#   V_max = maximum advection speed ||V|| = sqrt(u^2 + v^2)
#   N = polynomial degree (used as order parameter)

# Calculate characteristic length h_min
# For a triangle: h_min = 2*Area / max_perimeter_edge
# Here we use inscribed circle diameter: d_inscribed = 2*Area / semi-perimeter
area_element = jacobian / 2.0  # Element area (from Jacobian)

# Edge lengths
edge_lengths_calc = np.array([
    np.linalg.norm(v2 - v1),  # Edge 0: v1 v2
    np.linalg.norm(v3 - v2),  # Edge 1: v2 v3
    np.linalg.norm(v1 - v3)   # Edge 2: v3 v1
])
perimeter = np.sum(edge_lengths_calc)
semi_perimeter = perimeter / 2.0

# Inscribed circle diameter (most common choice for characteristic length in DG)
h_min = 2.0 * area_element / semi_perimeter

# Calculate maximum advection speed
u_field, v_field = velocity_field(x, y, t)
V_magnitude = np.sqrt(u_field**2 + v_field**2)
V_max = np.max(V_magnitude)

# Polynomial degree (order parameter N = k + 1 for DG)
N = k + 1

# Compute CFL-stable time step
dt = CFL * h_min / (V_max * N**2)

print(f"Time Integration Setup (DYNAMIC WAVE TEST):")
print(f"  Initial solution: q(x,y,0) = sin(x)")
print(f"  Final time: t_final = {t_final}")
print(f"  CFL = {CFL}")
print(f"\n  Dynamic time step calculation (CFL Stability):")
print(f"    Element area: {area_element:.6f}")
print(f"    Perimeter: {perimeter:.6f}, Semi-perimeter: {semi_perimeter:.6f}")
print(f"    Characteristic length (h_min): {h_min:.6f}")
print(f"    Max velocity (V_max): {V_max:.6f}")
print(f"    Polynomial degree (N = k+1): {N}")
print(f"    Formula: dt = CFL * h_min / (V_max * N^2)")
print(f"    Computed dt = {CFL:.1e} * {h_min:.6f} / ({V_max:.6f} * {N}^2)")
print(f"    dt = {dt:.6e}")

# ============================================================================
# STEP 2: LSRK54 Coefficients (Low Storage RK, 5-stage) - Exact Fractions
# ============================================================================

# Low storage RK54 (5-stage, 5th order): Carpenter & Kennedy (1994)
# Using exact fractional coefficients from blueprint (no decimal approximations)
A_RK = np.array([0.0,
                 -567301805773.0/1357537059087.0,
                 -2404267990393.0/2016746695238.0,
                 -3550918686646.0/2091501179385.0,
                 -1275806237668.0/842570457699.0])
B_RK = np.array([1432997174477.0/9575080441755.0,
                 5161836677717.0/13612068292357.0,
                 1720146321549.0/2090206949498.0,
                 3134564353537.0/4481467310338.0,
                 2277821191437.0/14882151754819.0])

print(f"\nLSRK54 Coefficients (exact fractions):")
print(f"  A_RK = {A_RK}")
print(f"  B_RK = {B_RK}")

# ============================================================================
# STEP 3: Initialize Residual Vector for Low-Storage RK
# ============================================================================

du = np.zeros_like(q)

# ============================================================================
# STEP 4: Time Stepping Loop (LSRK54) - REACHES t_final (NO EARLY EXIT)
# ============================================================================

l2_error_history = []
time_history = []
max_steps = 100000  # Allow many steps for dynamic simulation
step_tol = 1e-8

print(f"\nStarting time stepping loop...")
print(f"{'Step':<8} {'Time':>12} {'||q||_2':>15} {'L2_error':>15} {'dt':>12}")
print(f"{'-'*70}")

for step_count in range(max_steps):
    # Store error history
    q_exact_current = exact_solution(x, y, t)
    error_current = np.sqrt(np.sum((q - q_exact_current)**2 * M_diag))
    l2_error_history.append(error_current)
    time_history.append(t)

    # Print diagnostics
    q_norm = np.linalg.norm(q)
    if step_count % 50 == 0 or step_count < 5:
        print(f"{step_count:<8} {t:>12.6f} {q_norm:>15.6e} {error_current:>15.6e} {dt:>12.6e}")

    # Check for blow-up
    if np.isnan(q_norm) or np.isinf(q_norm) or q_norm > 1e10:
        print(f"❌ BLOW-UP detected at step {step_count}, t={t:.6f}")
        print(f"   ||q||_2 = {q_norm}")
        break

    # ========================================================================
    # CRITICAL: NO EARLY CONVERGENCE CHECK
    # For dynamic waves (transcendental functions), we ALWAYS reach t_final
    # The solution does NOT converge to steady state; waves propagate continuously.
    # ========================================================================

    # ========================================================================
    # LSRK54 Integration Step (5 stages, low-storage formulation)
    # ========================================================================

    # 動態步長檢查：確保最後一步「精確」踩在 t_final 上
    current_dt = min(dt, t_final - t)

    # Reset residual before the stages
    du.fill(0.0)

    for stage in range(5):
        # Compute RHS
        # diag = (step_count == 0 and stage == 0)
        R_q = compute_rhs(q, t, diagnostic=False)

        # LSRK54 Update using BOTH A_RK and B_RK coefficients
        # This is the correct low-storage RK54 formula:
        du = A_RK[stage] * du + current_dt * R_q
        q = q + B_RK[stage] * du

    t += current_dt

    # Check time stepping termination
    if t >= t_final:
        print(f"\n✓ Reached final time t = {t:.6f}")
        break

print(f"\nTime stepping completed at step {step_count}")
print(f"Final time: t = {t:.6f}")
print(f"Total steps: {step_count + 1}")

Time Integration Setup (DYNAMIC WAVE TEST):
  Initial solution: q(x,y,0) = sin(x)
  Final time: t_final = 1.0
  CFL = 0.1

  Dynamic time step calculation (CFL Stability):
    Element area: 0.500000
    Perimeter: 6.828427, Semi-perimeter: 3.414214
    Characteristic length (h_min): 0.292893
    Max velocity (V_max): 1.000000
    Polynomial degree (N = k+1): 5
    Formula: dt = CFL * h_min / (V_max * N^2)
    Computed dt = 1.0e-01 * 0.292893 / (1.000000 * 5^2)
    dt = 1.171573e-03

LSRK54 Coefficients (exact fractions):
  A_RK = [ 0.         -0.41789047 -1.19215169 -1.69778469 -1.51418344]
  B_RK = [0.14965902 0.37921031 0.82295503 0.69945046 0.15305725]

Starting time stepping loop...
Step             Time         ||q||_2        L2_error           dt
----------------------------------------------------------------------
0            0.000000    2.893164e+00    0.000000e+00 1.171573e-03
1            0.001172    2.891671e+00    6.783780e-06 1.171573e-03
2            0.002343    2.89017

## $L_2$ Error Calculation & Truncation Error Analysis

In [16]:
# Cell 5: L2 Error Analysis with Spatial Truncation Error Discussion

# ============================================================================
# STEP 0: Setup output directory
# ============================================================================

output_dir = Path(project_root) / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

# ============================================================================
# SPATIAL TRUNCATION ERROR ANALYSIS
# ============================================================================

print("\n" + "="*80)
print("SPATIAL TRUNCATION ERROR ANALYSIS FOR TRANSCENDENTAL FUNCTIONS")
print("="*80)

print(f"\n✓ Simulation completed at t = {t:.6f}")
print(f"\n📊 Final L2 Error Metrics:")
print(f"   L2 error at t={t:.6f}: {l2_error_history[-1]:.6e}")
print(f"   Maximum L2 error: {np.max(l2_error_history):.6e}")
print(f"   Minimum L2 error: {np.min(l2_error_history):.6e}")
print(f"   Average L2 error: {np.mean(l2_error_history):.6e}")

print(f"\n🔬 Theoretical Truncation Error (Sobolev Projection):")
print(f"   Exact solution: q_exact(x,y,t) = sin(x - t)  [TRANSCENDENTAL]")
print(f"   Approximation basis: Polynomials of degree k={k}  [(k+1)² = {(k+1)**2} nodes]")
print(f"   Characteristic mesh size: h_min = {h_min:.6f}")
print(f"   Polynomial degree: k = {k}")
print(f"   Expected convergence rate: O(h^{k+1}) = O(h^{k+1})")
print(f"\n   Estimated truncation error: C·h^{k+1} = C·({h_min:.6f})^{k+1}")
print(f"   (where C depends on solution smoothness and geometry)")

print(f"\n💡 Why NOT Machine Precision ($10^{{-15}}$)?")
print(f"\n   1. INTERPOLATION ERROR:")
print(f"      - sin(x-t) must be approximated by piecewise polynomials of degree {k}")
print(f"      - Even with high k, finite accuracy remains → O(h^{k+1}) error")
print(f"\n   2. NO EXACT REPRESENTATION:")
print(f"      - Unlike q=x-y (exactly representable in polynomial space),")
print(f"      - sin(x-t) requires infinite series → truncation error unavoidable")
print(f"\n   3. SPATIAL TRUNCATION DOMINATES:")
print(f"      - LSRK54 time integration: 5th-order in time")
print(f"      - DG spatial error: O(h^{k+1}) = O(h^4) for this test")
print(f"      - Since h = {h_min:.6f}, we have h^4 ≈ {h_min**4:.6e}")
print(f"      - Spatial error >> time integration error")

print(f"\n✅ This Test Validates Correct DG Implementation:")
print(f"   If error matches theoretical O(h^{k+1}) prediction → DG method is correct!")
print(f"   Perfect convergence (0.000e+00) would indicate overfitting or artifacts.")

print(f"\n" + "="*80)


SPATIAL TRUNCATION ERROR ANALYSIS FOR TRANSCENDENTAL FUNCTIONS

✓ Simulation completed at t = 1.000000

📊 Final L2 Error Metrics:
   L2 error at t=1.000000: 1.861844e-03
   Maximum L2 error: 1.861844e-03
   Minimum L2 error: 0.000000e+00
   Average L2 error: 8.528227e-04

🔬 Theoretical Truncation Error (Sobolev Projection):
   Exact solution: q_exact(x,y,t) = sin(x - t)  [TRANSCENDENTAL]
   Approximation basis: Polynomials of degree k=4  [(k+1)² = 25 nodes]
   Characteristic mesh size: h_min = 0.292893
   Polynomial degree: k = 4
   Expected convergence rate: O(h^5) = O(h^5)

   Estimated truncation error: C·h^5 = C·(0.292893)^5
   (where C depends on solution smoothness and geometry)

💡 Why NOT Machine Precision ($10^{-15}$)?

   1. INTERPOLATION ERROR:
      - sin(x-t) must be approximated by piecewise polynomials of degree 4
      - Even with high k, finite accuracy remains → O(h^5) error

   2. NO EXACT REPRESENTATION:
      - Unlike q=x-y (exactly representable in polynomial spac

In [17]:
import numpy as np
import pandas as pd
from IPython.display import display

def verify_surface_formulas(u_func, v_func, q_func, case_name):
    """
    數值解剖：檢查迎風通量 p 與 強形式表面積分 surface_term 的數值與維度。
    刻意加入 +1.0 的內部誤差，以觀察邊界機制的反應。
    """
    print("="*80)
    print(f"核心公式數值與陣列解剖: {case_name}")
    print("="*80)

    t = 0.0
    # 取得給定函數在物理網格上的值
    u, v = u_func(x, y, t), v_func(x, y, t)
    q_exact = q_func(x, y, t)

    # 刻意製造 +1.0 的內部誤差，讓公式有數值可以算
    q = q_exact + 1.0

    # ---------------------------------------------------------
    # 公式 1: p_i = 0.5 * (s_i - |s_i|) * (q_bound_i - q_g_i)
    # ---------------------------------------------------------
    q_bound = E @ q
    q_g = E @ q_exact

    normals_expanded = np.repeat(normals, nfp, axis=0)
    s = normals_expanded[:, 0] * u[0] + normals_expanded[:, 1] * v[0]

    # 計算 p (對應程式碼中的 flux_penalty)
    p = 0.5 * (s - np.abs(s)) * (q_bound - q_g)

    # 建立 DataFrame 1: 迎風懲罰通量 p
    p_reshaped = p.reshape(3, nfp)
    df_p = pd.DataFrame(
        p_reshaped,
        index=[f"Face {i}" for i in range(3)],
        columns=[f"Face Node {i}" for i in range(nfp)]
    )
    print(f"\\n[公式 1] 迎風懲罰通量 p")
    print(f"陣列形狀: {p.shape} (3 條邊 x {nfp} 個節點)")
    display(df_p.round(4))

    # ---------------------------------------------------------
    # 公式 2: surface_term = (1 / |T|) * W_inv @ E_T @ W_e @ p
    # ---------------------------------------------------------
    face_w = np.tile(weights_1d, 3)
    # Compute edge lengths and face Jacobian J_face = edge_length / 2.0
    face_i_indices = np.array([0, 1, 2])
    face_j_indices = np.array([1, 2, 0])
    edge_lengths = np.linalg.norm(vertices[face_j_indices] - vertices[face_i_indices], axis=1)
    J_face = edge_lengths / 2.0
    J_face_expanded = np.repeat(J_face, nfp)

    # 步驟 2.1: W_e @ p (包含邊界 1D 積分權重與 Face Jacobian)
    scaled_penalty = p * face_w * J_face_expanded

    # 步驟 2.2: E_T @ (W_e @ p) (將邊界通量放回 2D 體積節點)
    surface_term_weak = E.T @ scaled_penalty

    # 步驟 2.3: (1 / |T|) * W_inv 即為 M_inv_diag (乘上反質量矩陣)
    surface_term_strong = M_inv_diag * surface_term_weak

    # 建立 DataFrame 2: 強形式表面積分 surface_term
    is_affected = np.abs(surface_term_strong) > 1e-10
    df_surface = pd.DataFrame({
        "體積節點 (Volume Node)": np.arange(n_nodes),
        "強形式修正量 (Surface Term)": surface_term_strong,
        "受邊界影響": ["Yes" if flag else "" for flag in is_affected]
    })

    print(f"\\n[公式 2] 強形式表面積分 surface_term")
    print(f"陣列形狀: {surface_term_strong.shape} (對應 {n_nodes} 個全域體積節點)")

    # 高亮受影響的節點 (淡藍色背景)
    def highlight_affected(row):
        return ['background-color: #e6f2ff' if row['受邊界影響'] == 'Yes' else '' for _ in row]

    display(df_surface.round(4).style.apply(highlight_affected, axis=1))
    print("\\n\\n")


# ============================================================================
# 執行三種不同情境的測試
# ============================================================================

# Case 1: 靜態場 q = x - y, (u, v) = (1, 1) (往右上吹)
verify_surface_formulas(
    u_func=lambda x,y,t: np.ones_like(x),
    v_func=lambda x,y,t: np.ones_like(x),
    q_func=lambda x,y,t: x - y,
    case_name="u(x,y,t) = x - y, 風場 V = (1, 1)"
)

# Case 2: 動態 X-Wave q = sin(x - t), (u, v) = (1, 0) (純向右吹)
verify_surface_formulas(
    u_func=lambda x,y,t: np.ones_like(x),
    v_func=lambda x,y,t: np.zeros_like(x),
    q_func=lambda x,y,t: np.sin(x - t),
    case_name="u(x,y,t) = sin(x - t), 風場 V = (1, 0)"
)

# Case 3: 動態 Y-Wave q = sin(y - t), (u, v) = (0, 1) (純向上吹)
verify_surface_formulas(
    u_func=lambda x,y,t: np.zeros_like(x),
    v_func=lambda x,y,t: np.ones_like(x),
    q_func=lambda x,y,t: np.sin(y - t),
    case_name="u(x,y,t) = sin(y - t), 風場 V = (0, 1)"
)

核心公式數值與陣列解剖: u(x,y,t) = x - y, 風場 V = (1, 1)
\n[公式 1] 迎風懲罰通量 p
陣列形狀: (15,) (3 條邊 x 5 個節點)


,Face Node 0,Face Node 1,Face Node 2,Face Node 3,Face Node 4
Face 0,-1.0,-1.0,-1.0,-1.0,-1.0
Face 1,0.0,0.0,0.0,0.0,0.0
Face 2,-1.0,-1.0,-1.0,-1.0,-1.0


\n[公式 2] 強形式表面積分 surface_term
陣列形狀: (22,) (對應 22 個全域體積節點)


,體積節點 (Volume Node),強形式修正量 (Surface Term),受邊界影響
0,0,-35.890900,Yes
1,1,-35.890900,Yes
2,2,0.000000,
3,3,0.000000,
4,4,-35.890900,Yes
5,5,-35.890900,Yes
6,6,-23.313100,Yes
7,7,-23.313100,Yes
8,8,0.000000,
9,9,0.000000,


\n\n
核心公式數值與陣列解剖: u(x,y,t) = sin(x - t), 風場 V = (1, 0)
\n[公式 1] 迎風懲罰通量 p
陣列形狀: (15,) (3 條邊 x 5 個節點)


...

\n\n
核心公式數值與陣列解剖: u(x,y,t) = sin(y - t), 風場 V = (0, 1)
\n[公式 1] 迎風懲罰通量 p
陣列形狀: (15,) (3 條邊 x 5 個節點)


,Face Node 0,Face Node 1,Face Node 2,Face Node 3,Face Node 4
Face 0,-1.0,-1.0,-1.0,-1.0,-1.0
Face 1,0.0,0.0,0.0,0.0,0.0
Face 2,-0.0,-0.0,-0.0,-0.0,-0.0


\n[公式 2] 強形式表面積分 surface_term
陣列形狀: (22,) (對應 22 個全域體積節點)


,體積節點 (Volume Node),強形式修正量 (Surface Term),受邊界影響
0,0,-35.890900,Yes
1,1,0.000000,
2,2,0.000000,
3,3,0.000000,
4,4,0.000000,
5,5,-35.890900,Yes
6,6,-23.313100,Yes
7,7,0.000000,
8,8,0.000000,
9,9,0.000000,


\n\n
